In [1]:
!pip install --upgrade mediapipe==0.10.14 protobuf==4.25.3 -q

import os
import cv2
import numpy as np
import mediapipe as mp
import shutil
from glob import glob
from tqdm import tqdm
from pathlib import Path

BODY_LANDMARKS = [
    "nose", "leftEyeInner", "leftEye", "leftEyeOuter", "rightEyeInner", "rightEye", "rightEyeOuter",
    "leftEar", "rightEar", "mouthLeft", "mouthRight", "leftShoulder", "rightShoulder",
    "leftElbow", "rightElbow", "leftWrist", "rightWrist", "leftPinky", "rightPinky",
    "leftIndex", "rightIndex", "leftThumb", "rightThumb", 
    "leftHip", "rightHip", "leftKnee", "rightKnee", "leftAnkle", "rightAnkle", 
    "leftHeel", "rightHeel", "leftFootIndex", "rightFootIndex", "neck"
]

HAND_LANDMARKS = [
    "wrist", "indexTip", "indexDIP", "indexPIP", "indexMCP",
    "middleTip", "middleDIP", "middlePIP", "middleMCP",
    "ringTip", "ringDIP", "ringPIP", "ringMCP",
    "littleTip", "littleDIP", "littlePIP", "littleMCP",
    "thumbTip", "thumbIP", "thumbMP", "thumbCMC",
]

HANDS_LANDMARKS = [id + suffix for id in HAND_LANDMARKS for suffix in ["_0", "_1"]]
LANDMARKS = BODY_LANDMARKS + HANDS_LANDMARKS

print(f"✅ Đã cấu hình danh mục! Tổng: {len(LANDMARKS)} điểm khớp (34 Body + 42 Hands).")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 16.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-cloud-videointelligence 2.19.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 4.25.3 which is incompatible.
google-cloud-vision 3.14.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 4.25.3 which is incompatible.
a2a-sdk 0.3.26 requires protobuf>=5.29.5, but you have protobuf 4.25.3 which is incompatible.
grpc-google-iam-v1 0.14.4 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 4.25.3 which is incompatible.
google-cloud-dataplex 2.18.0 requires

2026-05-30 14:39:53.863473: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780151994.157759      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780151994.238769      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780151994.920819      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780151994.920881      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780151994.920884      16 computation_placer.cc:177] computation placer alr

✅ Đã cấu hình danh mục! Tổng: 76 điểm khớp (34 Body + 42 Hands).


In [2]:
import os
import cv2
import numpy as np
import mediapipe as mp

mp_holistic = mp.solutions.holistic
holistic = mp_holistic.Holistic(static_image_mode=False, model_complexity=1)
mp_drawing  = mp.solutions.drawing_utils

class SingleBodyDictNormalize:
    def __call__(self, row: dict) -> dict:
        sequence_size = len(row["leftEar"])
        ANCHOR_LANDMARKS = ["nose", "leftShoulder", "rightShoulder", "leftHip", "rightHip", "neck"]
        
        for i in range(sequence_size):
            x_coords = [row[name][i][0] for name in ANCHOR_LANDMARKS if row[name][i][0] != 0]
            y_coords = [row[name][i][1] for name in ANCHOR_LANDMARKS if row[name][i][1] != 0]
            
            if not x_coords or not y_coords: continue
            
            min_x, max_x = min(x_coords), max(x_coords)
            min_y, max_y = min(y_coords), max(y_coords)
            
            dx = (max_x - min_x) * 1.6
            dy = (max_y - min_y) * 1.6
            if dx <= 0 or dy <= 0: continue
            
            center_x = (max_x + min_x) / 2
            center_y = (max_y + min_y) / 2
            
            box_min_x = center_x - dx / 2
            box_min_y = center_y - dy / 2
            
            for key in BODY_LANDMARKS:
                x, y, z = row[key][i]
                if x == 0 and y == 0: continue  
                row[key][i] = ((x - box_min_x) / dx - 0.5, (y - box_min_y) / dy - 0.5, z)
        return row

class SingleHandDictNormalize:
    def __call__(self, row: dict) -> dict:
        sequence_size = len(row["leftEar"])
        for suffix in ["_0", "_1"]:
            for i in range(sequence_size):
                x_coords = [row[name + suffix][i][0] for name in HAND_LANDMARKS if row[name + suffix][i][0] != 0]
                y_coords = [row[name + suffix][i][1] for name in HAND_LANDMARKS if row[name + suffix][i][1] != 0]
                if not x_coords or not y_coords: continue
                min_x, max_x = min(x_coords), max(x_coords)
                min_y, max_y = min(y_coords), max(y_coords)
                dx, dy = max_x - min_x, max_y - min_y
                if dx <= 0 or dy <= 0: continue
                for name in HAND_LANDMARKS:
                    x, y, z = row[name + suffix][i]
                    if x == 0 and y == 0: continue
                    row[name + suffix][i] = ((x - min_x) / dx - 0.5, (y - min_y) / dy - 0.5, z)
        return row

body_norm = SingleBodyDictNormalize()
hand_norm = SingleHandDictNormalize()

def extract_sign_language_features(video_path, npy_out_path, visualize=False, vis_out_path=None):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened(): return False
    fps = cap.get(cv2.CAP_PROP_FPS)
    orig_w, orig_h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    video_writer = None
    if visualize and vis_out_path:
        pane_w = 400
        pane_h = int(pane_w * (orig_h / orig_w))
        os.makedirs(os.path.dirname(vis_out_path), exist_ok=True)
        video_writer = cv2.VideoWriter(vis_out_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (pane_w * 3, pane_h))
        
    video_sequence_dict = {name: [] for name in LANDMARKS}
    
    while True:
        ret, frame = cap.read()
        if not ret: break
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = holistic.process(rgb_frame)
        
        if visualize and video_writer:
            pane_original = cv2.resize(frame, (pane_w, pane_h))
            frame_skeleton_only = np.zeros_like(frame)
            frame_overlay = frame.copy()
            def draw_mp_skeleton(img):
                if results.pose_landmarks: mp_drawing.draw_landmarks(img, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS)
                if results.left_hand_landmarks: mp_drawing.draw_landmarks(img, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
                if results.right_hand_landmarks: mp_drawing.draw_landmarks(img, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
            draw_mp_skeleton(frame_skeleton_only); draw_mp_skeleton(frame_overlay)
            pane_skeleton = cv2.resize(frame_skeleton_only, (pane_w, pane_h))
            pane_overlay = cv2.resize(frame_overlay, (pane_w, pane_h))
            combined_pane = np.hstack((pane_original, pane_skeleton, pane_overlay))
            video_writer.write(combined_pane)
            
        pose_data = {name: (0.0, 0.0, 0.0) for name in BODY_LANDMARKS}
        if results.pose_landmarks:
            pose_map = {
                "nose": 0, "leftEyeInner": 1, "leftEye": 2, "leftEyeOuter": 3, "rightEyeInner": 4, "rightEye": 5, "rightEyeOuter": 6,
                "leftEar": 7, "rightEar": 8, "mouthLeft": 9, "mouthRight": 10, "leftShoulder": 11, "rightShoulder": 12,
                "leftElbow": 13, "rightElbow": 14, "leftWrist": 15, "rightWrist": 16, "leftPinky": 17, "rightPinky": 18,
                "leftIndex": 19, "rightIndex": 20, "leftThumb": 21, "rightThumb": 22, "leftHip": 23, "rightHip": 24,
                "leftKnee": 25, "rightKnee": 26, "leftAnkle": 27, "rightAnkle": 28, "leftHeel": 29, "rightHeel": 30,
                "leftFootIndex": 31, "rightFootIndex": 32
            }
            for name, idx in pose_map.items():
                lm = results.pose_landmarks.landmark[idx]
                pose_data[name] = (lm.x, lm.y, lm.z)
                
            ls, rs = pose_data["leftShoulder"], pose_data["rightShoulder"]
            pose_data["neck"] = ((ls[0] + rs[0])/2, (ls[1] + rs[1])/2, (ls[2] + rs[2])/2)
            
        for name in BODY_LANDMARKS: video_sequence_dict[name].append(pose_data[name])
            
        for suffix, hand_landmarks in [("_0", results.left_hand_landmarks), ("_1", results.right_hand_landmarks)]:
            hand_data = {name + suffix: (0.0, 0.0, 0.0) for name in HAND_LANDMARKS}
            if hand_landmarks:
                hand_map = {"wrist": 0, "thumbCMC": 1, "thumbMP": 2, "thumbIP": 3, "thumbTip": 4, "indexMCP": 5, "indexPIP": 6, "indexDIP": 7, "indexTip": 8, "middleMCP": 9, "middlePIP": 10, "middleDIP": 11, "middleTip": 12, "ringMCP": 13, "ringPIP": 14, "ringDIP": 15, "ringTip": 16, "littleMCP": 17, "littlePIP": 18, "littleDIP": 19, "littleTip": 20}
                for name, idx in hand_map.items():
                    lm = hand_landmarks.landmark[idx]
                    hand_data[name + suffix] = (lm.x, lm.y, lm.z)
            for name in HAND_LANDMARKS: video_sequence_dict[name + suffix].append(hand_data[name + suffix])
                
    cap.release()
    if video_writer: video_writer.release()
        
    video_sequence_dict = body_norm(video_sequence_dict)
    video_sequence_dict = hand_norm(video_sequence_dict)
    
    frames_list = []
    seq_len = len(video_sequence_dict["neck"])
    if seq_len == 0: return False
    for i in range(seq_len):
        frame_features = [video_sequence_dict[name][i] for name in LANDMARKS]
        frames_list.append(frame_features)
        
    np_array_data = np.array(frames_list, dtype=np.float32)
    os.makedirs(os.path.dirname(npy_out_path), exist_ok=True)
    np.save(npy_out_path, np_array_data)
    return True

print("✅ Pipeline core trích xuất 3D hoạt động ổn định!")

✅ Pipeline core trích xuất 3D hoạt động ổn định!


In [3]:
import os
import shutil
import numpy as np

# THIẾT LẬP LƯỢT CHẠY TẠI ĐÂY (Nhập giá trị: 1, 2, 3, hoặc 4)
CURRENT_BATCH = 4

KAGGLE_INPUT_DIR = "/kaggle/input/datasets/nguyenanfms/vsl-vietnamese-sign-language-v2/VSL_FULL_FRONT/VSL_FULL_FRONT"
JSON_PATH = "/kaggle/input/datasets/nguyenanfms/vsl-vietnamese-sign-language-v2/vsl_full_front.json"
OUTPUT_KEYPOINTS_DIR = "/kaggle/working/keypoints"

if os.path.exists(OUTPUT_KEYPOINTS_DIR):
    shutil.rmtree(OUTPUT_KEYPOINTS_DIR)
    print("🧹 Đã dọn sạch dữ liệu cũ lưu trên đĩa.")


if os.path.exists('/kaggle/working'):
    for item in os.listdir('/kaggle/working'):
        item_path = os.path.join('/kaggle/working', item)
        if not item.startswith('.'):
            if os.path.isdir(item_path): shutil.rmtree(item_path)
            else: os.remove(item_path)

all_classes = sorted([f for f in os.listdir(KAGGLE_INPUT_DIR) if os.path.isdir(os.path.join(KAGGLE_INPUT_DIR, f))])
total_classes = len(all_classes)
chunks = np.array_split(all_classes, 4)
active_classes = list(chunks[CURRENT_BATCH - 1])

print(f"📊 Dataset gốc: {total_classes} classes.")
print(f"🚀 TIẾN TRÌNH ĐANG XỬ LÝ: BATCH SỐ {CURRENT_BATCH}/4")
print(f"🔥 Số lượng danh mục phụ trách: {len(active_classes)} classes (Từ '{active_classes[0]}' đến '{active_classes[-1]}').")

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1780152012.760118      73 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1780152012.784373      73 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1780152012.788603      73 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1780152012.788608      71 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1780152012.789227      74 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1780152012.795797     

📊 Dataset gốc: 472 classes.
🚀 TIẾN TRÌNH ĐANG XỬ LÝ: BATCH SỐ 4/4
🔥 Số lượng danh mục phụ trách: 118 classes (Từ 'Tháng mười hai' đến 'Ủng hộ').


In [4]:
from glob import glob
from tqdm import tqdm
from pathlib import Path
import os

for class_name in active_classes:
    class_input_path = os.path.join(KAGGLE_INPUT_DIR, class_name)
    video_files = glob(os.path.join(class_input_path, "*.mp4"))
    print(f"\n📂 Tiến hành trích xuất: '{class_name}' ({len(video_files)} videos)")
    
    for video_path in tqdm(video_files, desc=f"Batch {CURRENT_BATCH} -> {class_name}"):
        video_id = Path(video_path).stem
        npy_out_path = os.path.join(OUTPUT_KEYPOINTS_DIR, class_name, f"{video_id}.npy")
        
        if os.path.exists(npy_out_path):
            continue
            
        extract_sign_language_features(video_path, npy_out_path, visualize=False)

print(f"\n🎉 HOÀN THÀNH XỬ LÝ TOÀN BỘ BATCH SỐ {CURRENT_BATCH}/4!")


📂 Tiến hành trích xuất: 'Tháng mười hai' (64 videos)


Batch 4 -> Tháng mười hai:   0%|          | 0/64 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
Batch 4 -> Tháng mười hai: 100%|██████████| 64/64 [04:52<00:00,  4.57s/it]



📂 Tiến hành trích xuất: 'Tháng mười một' (63 videos)


Batch 4 -> Tháng mười một: 100%|██████████| 63/63 [04:51<00:00,  4.62s/it]



📂 Tiến hành trích xuất: 'Tháng một' (61 videos)


...


📂 Tiến hành trích xuất: 'Ồn ào' (63 videos)


Batch 4 -> Ồn ào: 100%|██████████| 63/63 [03:51<00:00,  3.67s/it]



📂 Tiến hành trích xuất: 'Ủng hộ' (4 videos)


Batch 4 -> Ủng hộ: 100%|██████████| 4/4 [00:07<00:00,  1.77s/it]


🎉 HOÀN THÀNH XỬ LÝ TOÀN BỘ BATCH SỐ 4/4!


In [5]:
import os
print("📦 Đang nén toàn bộ thư mục keypoints thành file keypoints.zip...")
!zip -rq /kaggle/working/keypoints.zip /kaggle/working/keypoints
print("✅ Hoàn tất nén! Bạn có thể tải trực tiếp file keypoints.zip từ mục Output của Kaggle.")

📦 Đang nén toàn bộ thư mục keypoints thành file keypoints.zip...
✅ Hoàn tất nén! Bạn có thể tải trực tiếp file keypoints.zip từ mục Output của Kaggle.
